In [ ]:
#| default_exp search

# search

> web search for AI agents — text, images, news & videos via ddgs, plus direct Google via a stealth browser

fossick searches the web through [`ddgs`](https://github.com/deedy5/ddgs), a metasearch library that aggregates
many backends (bing, brave, duckduckgo, startpage, mojeek, yandex, wikipedia, …) with no API key and no Docker.
`search()` returns text results; `images()`, `news()` and `videos()` return the corresponding media.

Direct Google is soft-blocked for many IPs (plain HTTP gets a JavaScript-only shell), so `google()` uses scrapling's
stealth browser to get real Google ranking when you specifically need it. Every result is a plain dict, and all
functions share a local TTL cache so repeated queries return instantly.

In [ ]:
#| export
from urllib.parse import quote_plus
from fastcore.all import first, bind, L
from scrapling import Selector
from diskcache import Cache
from ddgs import DDGS
from primp import Client
from fossick.core import to_md, fetch, fetch_all, fossick_cache

## Text, image, news & video search

In [ ]:
#| export
_ddgs = None
def get_ddgs():
	global _ddgs
	if _ddgs is None: _ddgs = DDGS(verify=False)
	return _ddgs

sc = Cache(str(fossick_cache('search_cache.db')))
@sc.memoize(expire=900)
def search(q:str, category:str='text', **kw) -> list:
    "Search for text/images/news/videos/books via ddgs. Returns dicts with category-specific fields."
    if not (q and q.strip()): return []
    assert category in ('text', 'images', 'news', 'videos', 'books'), f'Invalid category: {category}'
    return getattr(get_ddgs(), category)(q, **kw)

text = bind(search, category='text')
images = bind(search, category='images')
news = bind(search, category='news')
videos = bind(search, category='videos')
books = bind(search, category='books')

def extract(url, **kw): return get_ddgs().extract(url, **kw)

## Google (stealth browser)

Direct Google is soft-blocked for many IPs — plain HTTP (curl/requests, and ddgs's own `google` backend) gets a
JavaScript-only shell with zero parseable results, regardless of consent cookies or UA. `google()` sidesteps this with
scrapling's **stealth browser** (`fetch(..., stealthy=True)`): it executes the page JS and evades bot detection, so it
gets a real SERP. It's slower (a browser launches per query), so it's an opt-in source for when you need Google's own
ranking — results are TTL-cached to amortise the cost.

In [ ]:
#| export
def _parse_google(html:str, n:int) -> list:
    "Parse organic results from a rendered Google SERP into dicts (title, url, content)."
    sel = Selector(html)
    out, seen = [], set()
    for h3 in sel.css('h3'):
        a = h3.parent
        while a is not None and a.tag != 'a': a = a.parent          # nearest ancestor anchor
        if a is None: continue
        href = a.attrib.get('href', '')
        if not href.startswith('http') or 'google.' in href or href in seen: continue
        box = a
        for _ in range(4): box = box.parent if box.parent is not None else box   # climb to result container
        snip = first(box.css('div.VwiC3b'))                         # Google's snippet block
        seen.add(href)
        out.append(dict(title=h3.get_all_text().strip(), href=href,
                        content=snip.get_all_text().strip() if snip else '', score=float(n - len(out))))
        if len(out) >= n: break
    return out

def _google(q:str, n:int, lang:str) -> list:
    "Stealth-browser Google fetch+parse, memoized. Raises on empty so throttled/failed calls aren't cached."
    pg = fetch(f'https://www.google.com/search?q={quote_plus(q)}&hl={lang}&num={n+2}',
               stealthy=True, network_idle=True, verify=False)
    res = _parse_google(pg.html_content, n) if pg.status == 200 else []
    if not res: raise ValueError('no google results')
    return res

@sc.memoize(expire=900)
def google(q:str, n:int=10, lang:str='en') -> list:
    "Search Google directly via a stealth browser (real Google ranking). Returns dicts, same shape as search()."
    if not q or not q.strip(): return []
    try: return _google(q, n, lang)
    except Exception: return []

In [ ]:
#| export
def _first(d, *ks):
    for k in ks:
        if d.get(k): return d[k]
    return ''

def research(q:str,                 # search query
             n:int=5,               # number of top results to read
             engine:str='search',   # 'search' (ddgs metasearch) or 'google' (stealth browser ranking)
             sel:str=None,          # CSS selector to narrow each page before markdown
             chars:int=4000,        # max markdown chars kept per page
             auto:bool=True,        # auto-escalate fetching (plain->heavy->stealthy->session) per page
             region:str='us-en',
             **kw,                  # extra kwargs passed to fetch()
            ) -> dict:
    "Query -> read the top `n` results -> a cited markdown corpus. Returns {query, sources:[{title,href,md}], digest}."
    if not (q and q.strip()): return dict(query=q, sources=[], digest='')
    hits = google(q, n=n) if engine == 'google' else list(search(q, max_results=n, region=region))
    hits = hits[:n]
    urls = [h.get('href') or h.get('url') for h in hits]
    pages = fetch_all([u for u in urls if u], sel=sel, auto=auto, **kw)
    sources, parts = [], []
    for h, pg in zip([h for h, u in zip(hits, urls) if u], pages):
        href = h.get('href') or h.get('url', '')
        title = _first(h, 'title') or href
        try: md = to_md(pg, sel=sel)[:chars]
        except Exception: md = ''
        sources.append(dict(title=title, href=href, md=md))
        parts.append(f'## {title}\n{href}\n\n{md}')
    return dict(query=q, sources=sources, digest='\n\n---\n\n'.join(parts))

## Tests

In [ ]:
# empty query is a no-op for every entry point (no network)
assert search('') == []

# Google SERP parser works on a static fixture (no browser/network)
_serp = ('<html><body><div class="MjjYud"><div class="g">'
         '<a href="https://ex.com/p"><h3>Example Page</h3></a>'
         '<div class="VwiC3b">A short snippet.</div></div></div>'
         '<div><a href="https://www.google.com/aclk"><h3>Ad</h3></a></div></body></html>')
_g = _parse_google(_serp, 5)
assert len(_g) == 1, f'expected 1 organic result, got {len(_g)}'   # google.com ad link filtered out
assert _g[0]['href'] == 'https://ex.com/p' and _g[0]['title'] == 'Example Page'
assert _g[0]['content'] == 'A short snippet.'

In [ ]:
#| eval: false
# text search (ddgs metasearch — aggregates working backends, no Docker)
for r in search('fasthtml python web framework', n=5): print(r['title'] , '---', r['href'])

FastHTML making Python web app creation easier - Geeky Gadgets --- https://www.geeky-gadgets.com/fasthtml-python-framework/
Build Fast, Scalable Web Apps With Python Using FastHTML - --- https://ostechnix.com/build-web-apps-with-python-using-fasthtml/
Python Web Development Just Got Easier: Meet FastHTML, the --- https://jhhemal.medium.com/python-web-development-just-got-easier-meet-fasthtml-the-framework-youve-been-waiting-for-f881526ef0be
FastHTML - Modern web applications in pure Python --- https://fastht.ml/
FastHTML – Modern web applications in pure Python | Hacker --- https://news.ycombinator.com/item?id=41104305
FastHTML For Beginners: Build An UI to Python App in 5 Minutes --- https://www.bitdoze.com/fasthtml-start/
FastHTML: Modern web applications in pure Python – Answer.AI --- https://www.answer.ai/posts/2024-08-03-fasthtml.html
Working FastHTML (a new Python Web Framework) dev.nix - Project --- https://community.firebasestudio.dev/t/working-fasthtml-a-new-python-web-framewo

In [ ]:
#| eval: false
# image search
for r in search('red panda', 'images',n=3): print(r['title'], '—', r['image'])

Red Panda Perching on Tree during Daytime · Free Stock Photo — https://images.pexels.com/photos/146102/pexels-photo-146102.jpeg?cs=srgb&dl=animal-cute-leaves-146102.jpg&fm=jpg
Red Panda Facts for Kids | Red Pandas | Cute Red Panda Photos — https://animalfactguide.com/wp-content/uploads/2020/12/red_panda_220-scaled.jpg
Free Images : ai, head, red panda, eye, tree, botany, carnivore, branch ... — https://get.pxhere.com/photo/ai-head-red-panda-plant-eye-tree-botany-carnivore-branch-vegetation-organism-trunk-terrestrial-plant-biome-terrestrial-animal-snout-forest-whiskers-natural-material-fur-wilderness-temperate-broadleaf-and-mixed-forest-nature-reserve-jungle-wildlife-twig-plant-stem-woodland-Northern-hardwood-forest-old-growth-forest-tail-bear-1671788.jpg
Cub Red Panda Tree Branches Leaves Forest Background 4K HD Panda ... — https://www.hdwallpapers.in/download/cub_red_panda_tree_branches_leaves_forest_background_4k_hd_panda-HD.jpg
Red Panda Predators And Prey — https://earth.org/wp-con

In [ ]:
#| eval: false
# news search
for r in search('open source LLMs', 'news', n=3): print(r['date'], r['title'])

2026-06-20T10:03:19+00:00 Z.ai’s GLM-5.2 narrows gap with OpenAI and Anthropic
2023-11-26T13:00:00+00:00 Benefits of open source large language models vs proprietary (LLMs)
2025-03-19T13:00:00+00:00 The Open-Source LLM Revolution: Transforming Enterprise AI For A New Era
2026-07-01T10:03:19+00:00 Vector Institute releases UnBias-Plus, a free, open-source AI tool to detect and rewrite bias in text
2026-06-26T10:03:19+00:00 China's new open-source model accelerates AI hacking threat
2026-06-30T10:03:19+00:00 Ornith Is the Open-Source Coding Model Built for Agents, Not Humans
2026-06-14T10:03:19+00:00 Harvey Trains Open Source Models To Encode Law Firm Workflows
Opinion3 days ago AI Sovereignty: Taking Control of Your Legal Tech Future


In [ ]:
#| eval: false
for r in videos(q='aditya hrudayam', n=1): print(r)

{'title': 'Aditya Hridaya Stotra - with Sanskrit lyrics', 'content': 'https://www.youtube.com/watch?v=O00BFBR6hoM', 'description': 'Aditya Hridaya Stotram (also known as Aditya Hridayam or Aditya Hrudayam). With synchronized, on-screen lyrics. ~~~~~~~~~~~~~~~~ This is a hymn associated with Aditya (Lord Surya, the Sun God) and was recited by the sage Agastya to Lord Ram on the battlefield before His duel with the demon king Ravan. Agastya teaches Ram, who is fatigued after ...', 'duration': '10:06', 'embed_html': '<iframe width="1280" height="720" src="https://www.youtube.com/embed/O00BFBR6hoM?autoplay=1" frameborder="0" allowfullscreen></iframe>', 'embed_url': 'https://www.youtube.com/embed/O00BFBR6hoM?autoplay=1', 'image_token': 'd56bc6bfbd506ecde68d96830aedb502ebde4ccc187431597cceb84f2751543f', 'images': {'large': 'https://tse2.mm.bing.net/th/id/OVP.8XhX5E6wb7xAMdAX2JaENAHgFo?pid=Api', 'medium': 'https://tse2.mm.bing.net/th/id/OVP.8XhX5E6wb7xAMdAX2JaENAHgFo?pid=Api', 'motion': 'http

In [ ]:
#| eval: false
# real Google ranking via stealth browser (slower; use when you need Google specifically)
for r in google('vedicreader website', n=5): print(r['title'], '—', r['href'])

[2026-07-02 20:08:32] INFO: Fetched (200) <GET https://www.google.com/search?q=vedicreader+website&hl=en&num=7&sei=nThGao6UBrXh2roPg7KJoQ0> (referer: https://www.google.com/)


VedicReader — https://vedicreader.com/
vedicreader.com to help absorb vedic texts easily — https://www.reddit.com/r/AdvaitaVedanta/comments/1lpr29l/vedicreadercom_to_help_absorb_vedic_texts_easily/
Login - VedicReader — https://vedicreader.com/a/lgn
Vedic Heritage Portal — https://vedicheritage.gov.in/
Explore Hindu Scriptures, Ṛṣis, Purāṇas & Vedic Knowledge — https://vedicpeople.com/


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()